# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore, load, and process the FAIR^2 dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.github.io/croissant/spec/) available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset's metadata and explore its summary.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Loaded dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Let's review all available record sets and their fields. All entities are referenced via their `@id` per the Croissant standard.

In [ ]:
# List all record sets and their fields by @id

print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name if hasattr(record_set, 'name') else '-'}")
    print(f"  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id} | Name: {getattr(field, 'name', '-')}")
    print()

## 3. Data Extraction
Each record set can be loaded as a pandas DataFrame by its `@id`. 

Let's load all record sets and display the first few rows of the main clinical record set.


In [ ]:
# Extract all record sets, referencing by @id as required
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Fetch records as list of dicts
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set '{record_set_id}' with {len(df)} rows and {len(df.columns)} columns.")
    else:
        print(f"Record set '{record_set_id}' has no records.")

# Choose the primary record set for analysis:
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display_df = dataframes[main_record_set_id].head()
    display_df
else:
    print("No record sets loaded with records.")

## 4. Exploratory Data Analysis (EDA)
Let's process a numeric field (e.g., age at diagnosis) and perform basic EDA: filtering, normalization, and simple grouping. All field references are by their `@id`.

> You can review the complete field listing above to select suitable `@id` values for numeric and categorical fields. Here we illustrate the steps for representative fields.

In [ ]:
# Example setup -- modify these @ids according to your actual schema fields
df = dataframes[main_record_set_id].copy()
# Replace these with actual field @ids. For illustration, we'll use plausible identifiers:
possible_numeric_ids = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower()]
numeric_field_id = possible_numeric_ids[0] if possible_numeric_ids else None

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}")
    # Drop NA values for analysis
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df = df.assign(**{numeric_field_id: numeric_series})
    threshold = numeric_series.mean() # e.g. select one std above mean
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    normalized = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[numeric_field_id + "_normalized"] = normalized
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try to group by a possible categorical field; pick a field with 'sex' or 'location' in the @id
    group_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex','location','msi','histology'])]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped)
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found in the current record set.")

## 5. Visualization
Visualize data distributions or relationships between important clinical variables. We plot a histogram of a numeric field (age, interval, etc.) and a countplot for a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of main numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=12)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

# Plot barplot of main categorical field
if 'group_field_id' in locals() and group_field_id:
    plt.figure(figsize=(7,3))
    order = df[group_field_id].value_counts().index
    sns.countplot(data=df, x=group_field_id, order=order)
    plt.title(f'Counts by {group_field_id}')
    plt.show()

## 6. Conclusion
We demonstrated how to use `mlcroissant` to:
- Load Croissant-structured data by referencing record sets, fields, and columns by their `@id`.
- Review the dataset schema and its fields for structured exploration.
- Extract tabular data for analysis, perform common data cleaning steps, filtering, normalization, and grouping, all with references by `@id`.
- Visualize important clinical features.

This workflow makes your exploration robust, extensible, and reproducible per the FAIR data principles.